# Tách keyframe 7 video S01 cuối (Google Colab)

Notebook tự lo hết: **tải video từ Hugging Face → tách keyframe → nén zip → upload kết quả lại Hugging Face → xoá file tạm**, lần lượt từng video.
Bạn chỉ cần: chọn GPU, đăng nhập Hugging Face, bấm chạy các cell theo thứ tự.

**Trước khi chạy:** menu `Runtime → Change runtime type → A100 GPU` (GPU nào cũng chạy được; A100 40GB dư VRAM, nút thắt là số CPU chứ không phải GPU).

| Cell | Việc | Ghi chú |
|---|---|---|
| 1 | Cài thư viện + kiểm tra GPU | ~1 phút |
| 2 | Đăng nhập Hugging Face | dán token có quyền **Write** vào ô hiện ra |
| 3 | Cấu hình | `VIDEOS` (đã điền sẵn 7 video) và `NUM_WORKERS` (số video chạy song song, tự chọn theo số CPU) |
| 4 | Nạp model + thuật toán | giống hệt pipeline local |
| 5 | Chạy toàn bộ | chạy `NUM_WORKERS` video song song, mỗi dòng log có tiền tố `[tên video]` kèm fps thực tế |

**Video AV1:** một số video S01 (V004…V011) mã hóa bằng AV1, OpenCV trên Colab không đọc được. Notebook tự chuyển sang PyAV (bộ giải mã dav1d, nhanh hơn) và cho ảnh giống hệt OpenCV. Video H.264 (V012) vẫn dùng OpenCV.

**Kết quả** nằm ở `5clnl/idk` → thư mục `keyframes_S01/`: mỗi video có `<tên>.zip` (ảnh keyframe, tên dạng `S01-V004_frame_012345.jpg`) và `<tên>.json` (thống kê).

**Bị ngắt giữa chừng (Colab hết phiên, mất mạng…)?** Chạy lại từ cell 1: video nào đã có `.zip` trên Hugging Face sẽ tự được bỏ qua. Video đang dở thì làm lại từ đầu.


In [ ]:
#@title 1. Cài thư viện & kiểm tra GPU
# Ghim đúng phiên bản open_clip đang dùng ở máy local để kết quả tách keyframe khớp với phần đã làm.
!pip install -q "open_clip_torch==3.3.0" || pip install -q open_clip_torch
!pip install -q av

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Không có GPU. Vào Runtime > Change runtime type > chọn GPU (A100), rồi chạy lại từ cell 1.")
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name}, {p.total_memory/1e9:.0f} GB VRAM")
import av
print(f"PyAV {av.__version__} | bộ giải mã AV1 (libdav1d): {'có' if 'libdav1d' in av.codecs_available else 'KHÔNG CÓ -> video AV1 sẽ không đọc được'}")


In [ ]:
#@title 2. Đăng nhập Hugging Face
# Hiện ô nhập token: dán token có quyền Write (huggingface.co/settings/tokens).
from huggingface_hub import login, whoami
login()
print("Đã đăng nhập với tài khoản:", whoami()["name"])


In [ ]:
#@title 3. Cấu hình
import os
SOURCE_REPO = "5clnl/idk"          # nơi lấy video (thư mục videos/)
RESULT_REPO = "5clnl/idk"          # nơi ghi kết quả
RESULT_DIR  = "keyframes_S01"      # thư mục kết quả trong repo (không đụng tới keyframes.zip có sẵn)

# Video cần tách (tên file đầy đủ, nằm trong videos/ của SOURCE_REPO). Muốn chạy video khác thì sửa danh sách này.
VIDEOS = [
    "S01-V004.mp4",
    "S01-V005.mp4",
    "S01-V008.mp4",
    "S01-V009.mp4",
    "S01-V010.mp4",
    "S01-V011.mp4",
    "S01-V012.mp4",
]

# Cách đọc video: "auto" = thử OpenCV trước, không đọc được (vd. video AV1) thì chuyển sang PyAV. Có thể ép "cv2" hoặc "av".
DECODER = "auto"

# Số video xử lý song song. Giải mã video chạy bằng CPU nên số này nên theo số CPU, không phải VRAM
# (các luồng dùng chung 1 bản model trên GPU; mỗi luồng chỉ thêm ~2GB VRAM và ~2.5GB RAM).
# Mặc định = nửa số CPU của máy Colab. Muốn tự chọn thì gán số cụ thể, ví dụ NUM_WORKERS = 4.
# Nếu fps của từng video tụt mạnh so với lúc chạy 1 video thì giảm xuống; nếu CPU còn rảnh thì tăng lên.
NUM_WORKERS = min(len(VIDEOS), max(1, (os.cpu_count() or 2) // 2))
print(f"Sẽ xử lý {len(VIDEOS)} video: {', '.join(VIDEOS)}")
print(f"Máy có {os.cpu_count()} CPU -> chạy {NUM_WORKERS} video song song")


In [ ]:
#@title 4. Nạp model & thuật toán tách keyframe (giống hệt pipeline local)
#@markdown CLIP ViT-L-14 (dfn2b), fp16 · ngưỡng 0.6 · cách nhau tối thiểu 5 frame · lấy mẫu mỗi 5 frame · so với keyframe gần nhất đã chọn.
import os, time, queue, threading
import cv2, numpy as np, torch, open_clip
from PIL import Image
try:
    import av
except Exception:
    av = None

THRESHOLD, MIN_DELTA, SAMPLE_RATE = 0.6, 5, 5
BATCH_SIZE = 128
QUEUE_MAX = 256
AV_THREADS = 0   # số luồng giải mã của PyAV cho mỗi video (0 = tự chọn); cell 5 đặt lại theo số video song song

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang nạp CLIP ViT-L-14 (dfn2b) lên {device}...")
model, _, preprocess = open_clip.create_model_and_transforms(model_name="ViT-L-14", pretrained="dfn2b", device="cpu")
if device == "cuda":
    model = model.half()
model = model.to(device).eval()
print("Model sẵn sàng.")


@torch.inference_mode()
def encode(frames_bgr):
    tensors = [preprocess(Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))) for f in frames_bgr]
    batch = torch.stack(tensors).to(device)
    if device == "cuda":
        batch = batch.half()
    feats = model.encode_image(batch)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy().astype(np.float32)


def _put(q, item, stop):
    while not stop.is_set():
        try:
            q.put(item, timeout=1)
            return True
        except queue.Full:
            continue
    return False


def _frames_cv2(path):
    # Giải mã mọi frame nhưng chỉ chuyển màu (retrieve) frame được lấy mẫu: grab() bỏ qua bước đó,
    # nhanh gấp ~2 lần mà frame lấy mẫu vẫn giống hệt từng pixel so với read().
    cap = cv2.VideoCapture(path)
    try:
        idx = 0
        while True:
            if idx % SAMPLE_RATE == 0:
                ok, frame = cap.read()
                if not ok:
                    return
                yield idx, frame
            else:
                if not cap.grab():
                    return
            idx += 1
    finally:
        cap.release()


def _frames_av(path):
    if av is None:
        raise RuntimeError("chưa cài PyAV (pip install av)")
    with av.open(path) as container:
        stream = container.streams.video[0]
        stream.thread_type = "AUTO"
        if AV_THREADS:
            stream.codec_context.thread_count = AV_THREADS
        for idx, fr in enumerate(container.decode(stream)):
            if idx % SAMPLE_RATE == 0:
                # src_colorspace="ITU601" là cách OpenCV chuyển YUV -> BGR (đã đối chiếu trên video thật: sai lệch 0.000),
                # nên ảnh từ PyAV giống hệt ảnh từ OpenCV.
                yield idx, fr.reformat(format="bgr24", src_colorspace="ITU601").to_ndarray()


def _codec_name(path):
    try:
        with av.open(path) as c:
            return c.streams.video[0].codec_context.name
    except Exception as e:
        return f"? ({type(e).__name__})"


def _probe(path):
    """(tổng số frame, fps) — thử OpenCV rồi PyAV; chỉ để hiển thị tiến độ."""
    total, fps = 0, 0.0
    cap = cv2.VideoCapture(path)
    if cap.isOpened():
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    cap.release()
    if total <= 0 and av is not None:
        try:
            with av.open(path) as c:
                st = c.streams.video[0]
                fps = float(st.average_rate or 0) or fps
                total = int(st.frames or (float(st.duration * st.time_base) * fps if st.duration else 0))
        except Exception:
            pass
    return total, (fps or 25.0)


def _reader(path, q, stop, info):
    order = {"auto": ["cv2", "av"], "cv2": ["cv2"], "av": ["av"]}[DECODER]
    try:
        for name in order:
            gen = _frames_cv2(path) if name == "cv2" else _frames_av(path)
            try:
                first = next(gen)
            except StopIteration:
                info["log"].append(f"{name}: không đọc được frame nào")
                continue
            except Exception as e:
                info["log"].append(f"{name}: lỗi {type(e).__name__}: {e}")
                continue
            info["backend"] = name
            try:
                if _put(q, first, stop):
                    for item in gen:
                        if not _put(q, item, stop):
                            break
            except Exception as e:
                info["log"].append(f"{name}: dừng giữa chừng: {type(e).__name__}: {e}")
            finally:
                gen.close()
            break
    finally:
        _put(q, None, stop)


def extract_keyframes(video_path, out_dir):
    name = os.path.splitext(os.path.basename(video_path))[0]
    total, src_fps = _probe(video_path)

    info = {"backend": None, "log": []}
    q = queue.Queue(maxsize=QUEUE_MAX)
    stop = threading.Event()
    th = threading.Thread(target=_reader, args=(video_path, q, stop, info), daemon=True)
    th.start()

    state = {"prev": None, "last_kf": -MIN_DELTA, "saved": 0, "sampled": 0, "last_idx": 0}

    def flush(indices, frames):
        feats = encode(frames)
        for k, idx in enumerate(indices):
            f = feats[k]
            if state["prev"] is None:
                keep = True
            else:
                pn = float(np.linalg.norm(state["prev"]))
                diff = 0.0 if pn < 1e-8 else float(np.linalg.norm(f - state["prev"])) / pn
                keep = diff > THRESHOLD and (idx - state["last_kf"]) >= MIN_DELTA
            if keep:
                cv2.imwrite(os.path.join(out_dir, f"{name}_frame_{idx:06d}.jpg"), frames[k])
                state["prev"], state["last_kf"] = f, idx
                state["saved"] += 1

    t0 = time.time()
    t_print = t0
    b_idx, b_frames = [], []
    try:
        while True:
            item = q.get()
            if item is None:
                break
            idx, frame = item
            if state["sampled"] == 0 and info["log"]:
                print(f"[{name}] dùng bộ giải mã {info['backend']} ({'; '.join(info['log'])})", flush=True)
            b_idx.append(idx)
            b_frames.append(frame)
            state["last_idx"] = idx
            state["sampled"] += 1
            if len(b_frames) >= BATCH_SIZE:
                flush(b_idx, b_frames)
                b_idx, b_frames = [], []
                now = time.time()
                if now - t_print >= 30:
                    t_print = now
                    done = idx + 1
                    fps = done / (now - t0)
                    eta = (total - done) / fps / 60 if fps > 0 and total > 0 else float("nan")
                    print(f"[{name}] {done:,}/{total:,} frame ({100*done/max(total,1):.0f}%) · {fps:.0f} fps · {state['saved']} keyframe · còn ~{eta:.0f} phút", flush=True)
        if b_frames:
            flush(b_idx, b_frames)
    finally:
        stop.set()
        th.join(timeout=30)

    secs = time.time() - t0
    if state["saved"] == 0:
        raise RuntimeError("không đọc được frame nào (" + " | ".join(info["log"]) + f") codec={_codec_name(video_path)}")
    done = state["last_idx"] + 1
    return {
        "video": name,
        "decoder": info["backend"],
        "keyframes": state["saved"],
        "sampled_frames": state["sampled"],
        "frames_read": done,
        "frames_declared": total,
        "source_fps": src_fps,
        "seconds": round(secs, 1),
        "decode_fps": round(done / secs, 1) if secs > 0 else 0,
        "params": {"threshold": THRESHOLD, "min_delta": MIN_DELTA, "sample_rate": SAMPLE_RATE, "model": "ViT-L-14/dfn2b", "fp16": device == "cuda"},
    }


In [ ]:
#@title 5. Chạy: tải → tách keyframe → nén → upload → dọn (song song NUM_WORKERS video)
import json, os, shutil, time, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
WORK = Path("/content/work")
PENDING = Path("/content/results_pending")   # kết quả chưa upload được (nếu upload lỗi) sẽ giữ ở đây
WORK.mkdir(parents=True, exist_ok=True)
PENDING.mkdir(parents=True, exist_ok=True)


def log(stem, msg):
    print(f"[{stem}] {msg}", flush=True)


def upload_verified(local_path, path_in_repo, tries=3):
    stem, size = Path(path_in_repo).stem, Path(local_path).stat().st_size
    for t in range(1, tries + 1):
        try:
            api.upload_file(path_or_fileobj=str(local_path), path_in_repo=path_in_repo,
                            repo_id=RESULT_REPO, repo_type="dataset",
                            commit_message=f"Keyframes {stem} (Colab)")
            info = api.get_paths_info(RESULT_REPO, [path_in_repo], repo_type="dataset")
            if info and info[0].size == size:
                return True
            log(stem, f"upload xong nhưng kích thước không khớp (lần {t}/{tries})")
        except Exception as e:
            log(stem, f"upload lỗi (lần {t}/{tries}): {e}")
        time.sleep(10 * t)
    return False


remote = set(api.list_repo_files(RESULT_REPO, repo_type="dataset"))
todo = []
for f in VIDEOS:
    if f"{RESULT_DIR}/{Path(f).stem}.zip" in remote:
        print(f"[bỏ qua] {Path(f).stem}: đã có kết quả trên Hugging Face")
    else:
        todo.append(f)

# Video lớn nhất chạy trước, để video dài nhất không phải chạy một mình ở cuối.
sizes = {}
try:
    for i in api.get_paths_info(SOURCE_REPO, [f"videos/{f}" for f in todo], repo_type="dataset"):
        sizes[Path(i.path).name] = i.size
except Exception as e:
    print("(không lấy được dung lượng video, giữ nguyên thứ tự)", e)
todo.sort(key=lambda f: -sizes.get(f, 0))

# Các video tải về cùng lúc phải vừa ổ đĩa: giảm số luồng nếu cần.
workers = max(1, min(NUM_WORKERS, len(todo)))
free = shutil.disk_usage(WORK).free
while workers > 1 and sum(sorted(sizes.values(), reverse=True)[:workers]) * 1.15 + 3e9 > free:
    workers -= 1
AV_THREADS = max(2, (os.cpu_count() or 4) // workers)   # số luồng giải mã PyAV cho mỗi video
print(f"Chạy {len(todo)} video với {workers} luồng song song (ổ đĩa trống {free/1e9:.0f} GB, {os.cpu_count()} CPU)")

results, pending, failed = [], [], []


def process_one(fname):
    stem = Path(fname).stem
    zip_repo, json_repo = f"{RESULT_DIR}/{stem}.zip", f"{RESULT_DIR}/{stem}.json"
    t0 = time.time()
    try:
        video_path = hf_hub_download(repo_id=SOURCE_REPO, repo_type="dataset", filename=f"videos/{fname}", local_dir=str(WORK))
        log(stem, f"đã tải xong ({os.path.getsize(video_path)/1e9:.2f} GB, {time.time()-t0:.0f}s)")

        kf_dir = WORK / "kf" / stem
        if kf_dir.exists():
            shutil.rmtree(kf_dir)
        kf_dir.mkdir(parents=True)

        stats = extract_keyframes(video_path, str(kf_dir))
        if stats["keyframes"] == 0:
            raise RuntimeError("không ra keyframe nào (video không đọc được?)")
        log(stem, f"xong: {stats['keyframes']} keyframe / {stats['frames_read']:,} frame trong {stats['seconds']/60:.1f} phút ({stats['decode_fps']} fps, {stats['decoder']})")

        zip_path, json_path = PENDING / f"{stem}.zip", PENDING / f"{stem}.json"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as zf:
            for p in sorted(kf_dir.glob("*.jpg")):
                zf.write(p, arcname=p.name)
        json_path.write_text(json.dumps(stats, ensure_ascii=False, indent=1), encoding="utf-8")

        # .json lên trước, .zip lên sau: sự có mặt của .zip trên HF = video này đã xong hoàn toàn.
        if upload_verified(json_path, json_repo) and upload_verified(zip_path, zip_repo):
            log(stem, f"đã upload lên {RESULT_REPO}/{RESULT_DIR}/")
            zip_path.unlink(); json_path.unlink()
            results.append(stats)
        else:
            log(stem, f"!! upload thất bại — giữ {zip_path} để tải về thủ công")
            pending.append(stem)
    except Exception as e:
        log(stem, f"!! LỖI: {e}")
        failed.append(stem)
    finally:
        shutil.rmtree(WORK / "kf" / stem, ignore_errors=True)
        (WORK / "videos" / fname).unlink(missing_ok=True)


with ThreadPoolExecutor(max_workers=workers) as ex:
    list(ex.map(process_one, todo))

print("\n================ TỔNG KẾT ================")
for s in sorted(results, key=lambda x: x["video"]):
    print(f"  OK  {s['video']}: {s['keyframes']} keyframe, {s['seconds']/60:.1f} phút")
if pending:
    print("  Chưa upload được (còn trong /content/results_pending):", ", ".join(sorted(pending)))
    print("  -> tải về máy bằng: from google.colab import files; files.download('/content/results_pending/<tên>.zip')")
if failed:
    print("  Lỗi (chạy lại cell này để thử lại):", ", ".join(sorted(failed)))
if not (pending or failed):
    print("  Tất cả đã xong.")
